In [2]:
# Core
import torch
import numpy as np

# Models
from models.hf_model import HFModel
from models.outputs import ModelOutput

# Uncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.black_uncertainty import BlackBoxUncertainty

# Evaluation
from evaluation.aggregation import Aggregator
from evaluation.hallucination_score import HallucinationScore
from evaluation.thresholds import HallucinationThresholds
from evaluation.report import EvaluationReport

# Decision
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider


In [3]:
model = HFModel("EleutherAI/gpt-neo-125M")

  # token yoksa open model

prompt = "What is the capital city of Turkey?"

output = model.generate(
    prompt,
    max_new_tokens=10,
    num_return_sequences=3,
)

output.responses


Loading weights: 100%|██████████| 160/160 [00:02<00:00, 55.90it/s, Materializing param=transformer.wte.weight]                          
GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['What is the capital city of Turkey?\n\n\n\nKaraıl, Ankara,',
 'What is the capital city of Turkey?” She says, “Is it the',
 'What is the capital city of Turkey?\nTo get a glimpse into the political reality of']

In [4]:
if output.has_whitebox():
    white = WhiteBoxUncertainty(
        scores=output.logits,
        token_ids=output.token_ids,
        text_responses=output.responses
    )

    white_entropy = white.predictive_entropy()
    white_conf = white.confidence()
    white_cons = white.self_consistency()

    print("White Entropy:", white_entropy)
    print("White Confidence:", white_conf)
    print("White Consistency:", white_cons)
else:
    white_entropy = None


White Entropy: 2.222975015640259
White Confidence: 2.121310903885934e-13
White Consistency: 0.3333333333333333


In [5]:
gray = GrayBoxUncertainty(
    responses=output.responses,
    log_probs=output.log_probs
)

gray_conf = gray.confidence()
gray_entropy = gray.response_entropy()

print("Gray Confidence:", gray_conf)
print("Gray Entropy:", gray_entropy)


Gray Confidence: 0.02525914746957436
Gray Entropy: 1.0986122886651097


In [6]:
black = BlackBoxUncertainty(output.responses)

black_conf = black.confidence()
black_entropy = black.response_entropy()
black_unique = black.unique_ratio()

print("Black Confidence:", black_conf)
print("Black Entropy:", black_entropy)
print("Black Unique Ratio:", black_unique)


Black Confidence: 0.3333333333333333
Black Entropy: 1.0986122886651097
Black Unique Ratio: 1.0


In [7]:
consistency = Aggregator.consistency_score(output.responses)
unique_ratio = Aggregator.unique_ratio(output.responses)

print("Aggregator Consistency:", consistency)
print("Aggregator Unique Ratio:", unique_ratio)


Aggregator Consistency: 0.3333333333333333
Aggregator Unique Ratio: 1.0


In [8]:
hs = HallucinationScore(
    white_score=white_entropy if output.has_whitebox() else None,
    gray_score=gray_conf,
    black_score=black_conf,
)

hallucination_score = hs.score()
hallucination_score


0.6702602570103484

In [9]:
risk_level = HallucinationThresholds.interpret(hallucination_score)
risk_level


'HIGH'

In [10]:
metrics = {
    "entropy": white_entropy if white_entropy is not None else 0,
    "self_consistency": black_conf,
    "confidence": gray_conf
}

final_score = FinalScore().compute(metrics)

decider = HallucinationDecider(
    thresholds={"hallucination": 0.6}
)

decision = decider.decide(
    {"final_score": final_score}
)

final_score, decision


(1.0275751690833517, 'hallucination')

In [11]:
report = EvaluationReport(
    hallucination_score=final_score,
    level=risk_level,
    white=white_entropy,
    gray=gray_conf,
    black=black_conf
)

report.pretty_print()
report.to_dict()


--- Evaluation Report ---
Hallucination Score : 1.0275751690833517
Risk Level : HIGH
White-box Entropy   : 2.2230
Gray-box Confidence : 0.0253
Black-box Consist.  : 0.3333


{'hallucination_score': 1.0275751690833517,
 'risk_level': 'HIGH',
 'white_uncertainty': 2.222975015640259,
 'gray_uncertainty': 0.02525914746957436,
 'black_uncertainty': 0.3333333333333333}

In [16]:
from pipeline.runner import PipelineRunner
from evaluation.evaluator import Evaluator

prompts = [
    "What is the capital of France?",
    "Explain photosynthesis.",
    "Who painted the Mona Lisa?",
    "Describe black holes.",
    "Define quantum entanglement."
]

# Model ve pipeline hazır
model = HFModel("gpt2")  # veya kendi local modelin
uncertainty_modules = {
    "blackbox": lambda output: BlackBoxUncertainty(output.responses)
}
evaluator = FinalScore()
decider = HallucinationDecider(thresholds={"hallucination": 0.7})

pipeline = PipelineRunner(model, uncertainty_modules, evaluator, decider)

# Çalıştır ve sonuçları sakla
results = []
for prompt in prompts:
    res = pipeline.run(prompt)
    results.append(res)
    print(f"Prompt: {prompt}")
    print(f"Responses: {res['responses']}")
    print(f"Uncertainty metrics: {res['uncertainty']}")
    print(f"Final evaluation: {res['evaluation']}")
    print(f"Decision: {res['decision']}")
    print("="*50)

# İstersen buradan scores ve groundtruth ile statistics/calibration hesaplayabilirsin
scores = [r['evaluation']['final_score'] for r in results]
labels = [0,0,0,1,1]  # Örnek groundtruth

from statistics import StatisticalAnalyzer
pearson, spearman = StatisticalAnalyzer.correlation(scores, labels)
auroc = StatisticalAnalyzer.auroc(scores, labels)
pr_auc = StatisticalAnalyzer.pr_auc(scores, labels)
print(f"Pearson: {pearson}, Spearman: {spearman}, AUROC: {auroc}, PR-AUC: {pr_auc}")

from calibration import CalibrationMetrics
brier = CalibrationMetrics.brier_score(scores, labels)
ece = CalibrationMetrics.expected_calibration_error(scores, labels)
print(f"Brier: {brier}, ECE: {ece}")


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 612.95it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


AttributeError: 'FinalScore' object has no attribute 'evaluate'

In [1]:
# ===========================
# IMPORTLAR
# ===========================
from models.hf_model import HFModel
from pipeline.runner import PipelineRunner
from evaluation.evaluator import Evaluator
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider
from uncertainty.black_uncertainty import BlackBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.semantic_uncertainty import EnsembleSemanticUncertainty

# NOT: statistics.py dosyanın adı 'stat_metrics.py' olarak değiştirildi
from stat_metrics import StatisticalAnalyzer
from calibration import CalibrationMetrics

# ===========================
# ÖRNEK PROMPTLAR
# ===========================
prompts = [
    "What is the capital of France?",
    "Explain photosynthesis.",
    "Who painted the Mona Lisa?",
    "Describe black holes.",
    "Define quantum entanglement."
]

# ===========================
# MODEL VE PIPELINE HAZIRLA
# ===========================
model = HFModel("gpt2")  # Local veya HF model

# Örnek uncertainty modules
uncertainty_modules = {
    "blackbox": lambda output: BlackBoxUncertainty(output.responses),
      "graybox": lambda output: GrayBoxUncertainty(output.responses, output.log_probs),
     "whitebox": lambda output: WhiteBoxUncertainty(output.logits, output.token_ids, output.responses),
    "semantic": lambda output: EnsembleSemanticUncertainty(output.responses, language="en")
}

# FinalScore -> Evaluator wrapper ile
final_score_calc = FinalScore()
evaluator = Evaluator(final_score_calc)

# HallucinationDecider
decider = HallucinationDecider(thresholds={"hallucination": 0.7})

# PipelineRunner
pipeline = PipelineRunner(model, uncertainty_modules, evaluator, decider)

# ===========================
# PIPELINE'İ ÇALIŞTIR VE SONUÇLARI AL
# ===========================
results = []

for prompt in prompts:
    res = pipeline.run(prompt)
    results.append(res)
    print(f"Prompt: {prompt}")
    print(f"Responses: {res['responses']}")
    print(f"Uncertainty metrics: {res['uncertainty']}")
    print(f"Final evaluation: {res['evaluation']}")
    print(f"Decision: {res['decision']}")
    print("="*50)

# ===========================
# SCORES VE GROUNDTRUTH İLE STATISTICS
# ===========================
scores = [r['evaluation']['final_score'] for r in results]
labels = [0,0,0,1,1]  # Örnek groundtruth: 1=hallucination, 0=reliable

pearson, spearman = StatisticalAnalyzer.correlation(scores, labels)
auroc = StatisticalAnalyzer.auroc(scores, labels)
pr_auc = StatisticalAnalyzer.pr_auc(scores, labels)

print("=== Statistical Metrics ===")
print(f"Pearson: {pearson:.4f}, Spearman: {spearman:.4f}, AUROC: {auroc:.4f}, PR-AUC: {pr_auc:.4f}")

# ===========================
# CALIBRATION METRICS
# ===========================
brier = CalibrationMetrics.brier_score(scores, labels)
ece = CalibrationMetrics.expected_calibration_error(scores, labels)

print("=== Calibration Metrics ===")
print(f"Brier Score: {brier:.4f}, Expected Calibration Error (ECE): {ece:.4f}")


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 295.80it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
/usr/local/python/3.12.1/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(
Loading weights: 100%|██████████|

Prompt: What is the capital of France?
Responses: ["What is the capital of France? What do you think the French do?\n\nI mean, it was a question that was not answered by anybody. And if we wanted to call ourselves 'French'. We did. They do things with words such as 'inconvenience',", 'What is the capital of France? Not, so far as the subject is concerned; but if we believe that capital is not as old as its name implies—that our first country is one in which, at the same time, it is endowed with more than sufficient resources—or that', 'What is the capital of France? Are these words being used by those who support Islamic fundamentalism?\n\n\nHow important is their language?\n\n\nHow important are those who believe Islam is a great and divine thing?\n\n\nHow important is Islam in these circumstances, if any?']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.025513

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 361.64it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 257.89it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 379.48it/s, Materializing param=poole

Prompt: Explain photosynthesis.
Responses: ['Explain photosynthesis.\n\nThe key was the discovery of an oxygen atom in the chlorides (H2O2) and hydrogen (H2O2) perchlorate (C2) molecules that was so important that their presence was required in laboratory experiments.', 'Explain photosynthesis. A simple recipe for cooking. Using ingredients for a homemade recipe. A quick method to prepare things for your family and friends. A new way of preparing things for people.\n\n"It was very hard to read the comments, and just a small', 'Explain photosynthesis.\n\n- It uses photodynamic (hydrogen) to drive the reaction of CO2 and CH4.\n\n- Can be used in cells but is not needed for long range.\n\n- Useful during the field.\n\n- Uses']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.028597698557086246, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': 2.409548

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 408.50it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 241.76it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 354.38it/s, Materializing param=poole

Prompt: Who painted the Mona Lisa?
Responses: ['Who painted the Mona Lisa?\n\nThey\'ve been so popular with her that they\'ve inspired many designs. One, "Hermit-Trees", is a lovely example of that.\n\nThey\'re like, "Yeah…they\'re good!" We did this, you', 'Who painted the Mona Lisa? Oh, don\'t worry, I did not paint it!"\n\n"You would be shocked to learn my exact face for any other colour! I have done it all before...I have painted all shades of hair in my entire life and all I', 'Who painted the Mona Lisa?\n\nWhen is it in print?\n\nWhere do you buy it?\n\nDoes it have color or a different colour?\n\nAnd much more.\n\n\nDid you know this song was written and performed by JL Armstrong?\n']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.027647522810492046, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': 2.702481269836426, 'white_confidence': 3.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 382.95it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 250.56it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 393.93it/s, Materializing param=poole

Prompt: Describe black holes.
Responses: ['Describe black holes.\n\nJULIA E. PUSSSON\n\nA scientist in the Department of Physics at the University of California, Berkeley on March 3, 2010. Photograph: Daniel J. DeYoung/U.C. Berkeley School of Medicine.\n', "Describe black holes. You could literally try and describe a black holes when you get to those. They could feel you in your own body, it wasn't like, like you're in outer space or something. Maybe then you couldn't tell if it was a black hole", "Describe black holes.\n\nThe term black hole refers to a black hole, the ultimate object, that the Universe can't possibly go through.\n\nThere is one explanation. Black holes are the ultimate manifestation of a black hole. That means they are the ultimate embodiment"]
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.03716732594708087, 'gray_entropy': np.float64(1.09861

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 356.00it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 281.37it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 379.37it/s, Materializing param=poole

Prompt: Define quantum entanglement.
Responses: ['Define quantum entanglement.\n\nAt the end of a computation, however, he writes that it has to happen as described for this, instead of all the problems that can arise in classical logic.\n\n"The point is that our proof that it has to do on', 'Define quantum entanglement. The idea was brought up by Gordon T. Miller, a Stanford University physicist now living in Paris and the coauthor of The Quantum Entanglement Hypothesis, to describe the idea of a quantum field as a state of affairs, based on a', 'Define quantum entanglement. This implies that information that can be transmitted through quantum channels is subject to state interference (the "stateless" information that is contained in stateless quantum vectors). This would also apply in conjunction to information whose quantum information can be decoded and transmitted (']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'b